# 04 — Robustez y sensibilidad del score RADAR | RADAR Cibest

**Fase ASUM-DM:** 5 — Evaluación  
**Responsable:** Jhon Farley Adarve Díaz  
**Fecha de ejecución:** Junio 2026  
**Propósito:** evaluar la robustez de rankings y prioridades país/línea bajo perturbaciones razonables de pesos, comparando sensibilidad determinística, TOPSIS vs VIKOR, Monte Carlo TOPSIS y Monte Carlo RADAR.

---

## 0. La decisión final debe validarse sobre RADAR, no solo sobre TOPSIS

Este notebook responde la pregunta de robustez ejecutiva:

> **¿Qué países mantienen su prioridad cuando se perturban razonablemente los pesos del modelo?**

El análisis distingue dos niveles:

1. **Robustez TOPSIS:** estabilidad del componente estructural ante incertidumbre en pesos de dimensiones y variables.
2. **Robustez RADAR:** estabilidad del score final ante incertidumbre en TOPSIS y en la mezcla `alpha`, `beta`, `gamma`.

La lectura ejecutiva debe privilegiar Monte Carlo RADAR, porque esa es la métrica de decisión final:

```text
RADAR = alpha * TOPSIS + beta * IPC + gamma * Trend
```

### Mensaje ejecutivo preliminar

Un país no debe considerarse prioritario solo por ocupar un lugar alto en el ranking base. Debe clasificarse según:

- ranking base;
- ranking medio Monte Carlo;
- volatilidad del ranking;
- probabilidad de pertenecer al Top-N;
- probabilidad de quedar en banda alta o media-alta;
- estabilidad de correlaciones entre líneas;
- sensibilidad de la decisión frente a cambios de pesos.

---

## 1. Configuración del entorno y estilo Cibest

In [1]:
# ---------------------------------------------------------------------------
# Configuración inicial del notebook
# ---------------------------------------------------------------------------
import sys
import re
from pathlib import Path
from typing import Optional, Union, Dict, List
import importlib
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display, Markdown

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))

import src
import src.scoring.hybrid_scorer
import src.scoring.sensitivity
import src.scoring.monte_carlo as monte_carlo

importlib.invalidate_caches()
importlib.reload(src.scoring.hybrid_scorer)
importlib.reload(src.scoring.sensitivity)
importlib.reload(monte_carlo)

from src.utils import (
    load_all_configs,
    get_variable_catalog,
    resolve_data_path,
    setup_logger,
)

from src.scoring.sensitivity import (
    run_sensitivity_analysis,
    compare_topsis_vikor,
)

from src.scoring.hybrid_scorer import (
    prepare_decision_matrix,
    run_full_scoring,
)

from src.scoring.weighting import compute_hierarchical_weights

from src.scoring.monte_carlo import (
    coerce_component_series,
    run_monte_carlo_topsis_robustness,
    run_monte_carlo_radar_robustness,
)

configs = load_all_configs()
setup_logger(configs["settings"].get("logging"))
catalog = get_variable_catalog(configs["variables"])

# ---------------------------------------------------------------------------
# Identidad visual Cibest
# ---------------------------------------------------------------------------
CIBEST = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
}

TIER_COLORS = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}

px.defaults.template = dict(
    layout=dict(
        font=dict(family="Arial, sans-serif", size=13, color=CIBEST["gray"]),
        title=dict(font=dict(size=17, color=CIBEST["gray"])),
        plot_bgcolor=CIBEST["white"],
        paper_bgcolor=CIBEST["white"],
        xaxis=dict(gridcolor=CIBEST["gray_border"], linecolor=CIBEST["gray"]),
        yaxis=dict(gridcolor=CIBEST["gray_border"], linecolor=CIBEST["gray"]),
        colorway=[CIBEST["gray"], CIBEST["gold"], CIBEST["green"], CIBEST["amber"]],
    )
)


def style_table(df, gradient_cols=None, gradient_cmap="YlGnBu", format_dict=None):
    """Aplica estilo Cibest a tablas pandas."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        styler = styler.background_gradient(subset=gradient_cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str):
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))

success_box("Entorno listo", "Configuración, módulos de sensibilidad y Monte Carlo cargados correctamente.")

---

## 2. Se carga el master vigente y se prepara la matriz de decisión

In [2]:
raw_dir = resolve_data_path(configs["settings"]["data"]["raw_path"])
pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

master_files = sorted(
    [
        path for path in raw_dir.glob("master_raw_*.parquet")
        if pattern.match(path.name)
    ],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

if not master_files:
    raise FileNotFoundError("Falta master_raw_YYYYMMDD.parquet. Ejecute primero notebook 01.")

master_path = master_files[0]
master = pd.read_parquet(master_path)

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master = master[master["variable"] != "wgi_composite"].copy()

master_summary = pd.DataFrame({
    "métrica": ["Archivo", "Filas", "Países", "Variables", "Tiene gdp_growth"],
    "valor": [
        master_path.name,
        master.shape[0],
        master["country_iso3"].nunique(),
        master["variable"].nunique(),
        "Sí" if "gdp_growth" in master["variable"].unique() else "No",
    ],
})

display(style_table(master_summary))

wide_raw, decision_matrix, excluded = prepare_decision_matrix(master, configs)

matrix_summary = pd.DataFrame({
    "métrica": ["wide_raw", "decision_matrix", "Países excluidos", "gdp_growth en decision_matrix"],
    "valor": [
        str(wide_raw.shape),
        str(decision_matrix.shape),
        ", ".join(excluded) if excluded else "Ninguno",
        "Sí" if "gdp_growth" in decision_matrix.columns else "No",
    ],
})

display(style_table(matrix_summary))

if "gdp_growth" in decision_matrix.columns:
    raise ValueError("gdp_growth no debe estar en decision_matrix. Debe alimentar solo el componente Trend.")

,métrica,valor
0,Archivo,master_raw_20260616.parquet
1,Filas,1288
2,Países,30
3,Variables,45
4,Tiene gdp_growth,Sí


2026-06-17 11:27:17 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-17 11:27:17 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-17 11:27:17 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-17 11:27:17 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,métrica,valor
0,wide_raw,"(25, 46)"
1,decision_matrix,"(25, 35)"
2,Países excluidos,"BRB, CUB, GUY, HTI, VEN"
3,gdp_growth en decision_matrix,No


**Lectura ejecutiva.** La sensibilidad debe ejecutarse sobre la misma matriz que alimenta scoring. `decision_matrix` debe excluir variables de IPC, auxiliares y `gdp_growth`, mientras `wide_raw` conserva insumos para auditoría, proximidad y tendencia.

---

## 3. Sensibilidad determinística de pesos: primera lectura de fragilidad estructural

Este análisis evalúa cómo cambia el ranking TOPSIS bajo perturbaciones de pesos configuradas en `src.scoring.sensitivity`.

In [3]:
sens = run_sensitivity_analysis(
    decision_matrix=decision_matrix,
    dimension_weights=configs["weights"]["dimension_weights"],
    variable_weights_by_dim=configs["weights"]["variable_weights"],
    variable_catalog=catalog,
)

display(style_table(sens if isinstance(sens, pd.DataFrame) else pd.DataFrame(sens)))

2026-06-17 11:27:18 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.703 (USA)


2026-06-17 11:27:18 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.705 (USA)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.704 (USA)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.703 (USA)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.702 (USA)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.713 (CAN)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.707 (USA)
2026-06-17 11:27:21 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.700 (USA)
2026-06-17 11:27:22 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.696 (USA)
2026-06-17 11:27:22 | INFO     | src.scoring.ranking:rank:133 | 

,dimension,perturbation,score_corr,topN_overlap,topN_size,countries_in,countries_out
0,macro,0.800000,0.993100,9,10,DOM,BRA
1,macro,0.900000,0.997700,9,10,DOM,BRA
2,macro,1.100000,0.993800,9,10,DOM,JAM
3,macro,1.200000,0.986200,9,10,MEX,JAM
4,financial,0.800000,0.989200,9,10,DOM,JAM
5,financial,0.900000,0.995400,9,10,DOM,JAM
6,financial,1.100000,0.999200,10,10,,
7,financial,1.200000,0.997700,10,10,,
8,institutional,0.800000,0.983800,9,10,MEX,JAM
9,institutional,0.900000,0.994600,10,10,,


**Interpretación.** Esta sensibilidad es útil como diagnóstico inicial, pero no reemplaza Monte Carlo. Sirve para identificar si el ranking estructural es muy sensible a cambios puntuales de pesos.

---

## 4. TOPSIS vs VIKOR: comparación de métodos multicriterio

In [4]:
var_weights = compute_hierarchical_weights(
    configs["weights"]["dimension_weights"],
    configs["weights"]["variable_weights"],
)

var_weights = {
    variable: weight
    for variable, weight in var_weights.items()
    if variable in decision_matrix.columns
}

comparison = compare_topsis_vikor(decision_matrix, var_weights, catalog)

display(style_table(
    comparison.head(20),
    gradient_cols=[col for col in comparison.columns if "rank" in col.lower() or "score" in col.lower()],
    gradient_cmap="RdYlGn",
))

2026-06-17 11:27:22 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 25 paises | score max=0.703 (USA)
2026-06-17 11:27:22 | INFO     | src.scoring.ranking:rank:180 | VIKOR completado: 25 paises | v=0.5
2026-06-17 11:27:22 | INFO     | src.scoring.sensitivity:compare_topsis_vikor:114 | TOPSIS vs VIKOR: correlacion Spearman = 0.932


,score_topsis,rank_topsis,score_vikor,rank_vikor,rank_diff
country_iso3,,,,,
USA,0.703436,1,1.000000,1,0
CAN,0.694204,2,0.983927,2,0
ESP,0.645020,3,0.951064,3,0
CHL,0.609601,4,0.729247,5,1
URY,0.582052,5,0.743622,4,1
CRI,0.546337,6,0.634115,7,1
BHS,0.530203,7,0.453165,11,4
PAN,0.515110,8,0.659829,6,2
JAM,0.501994,9,0.303140,17,8


**Lectura ejecutiva.** TOPSIS y VIKOR no optimizan exactamente la misma lógica. Si los rankings son similares, hay mayor robustez metodológica. Si divergen, los países afectados deben analizarse como decisiones sensibles al método multicriterio.

---

## 5. El scoring base genera IPC, Trend y ranking RADAR para usar como escenario central

In [5]:
results = run_full_scoring(master, configs, persist=True)

ipc_scores = coerce_component_series(
    results["ipc"],
    value_col=None,
    component_name="ipc",
)

trend_scores = coerce_component_series(
    results["trend"],
    value_col="trend",
    component_name="trend",
)

base_summary = pd.DataFrame({
    "métrica": ["Países RADAR", "IPC disponible", "Trend disponible", "Alpha", "Beta", "Gamma"],
    "valor": [
        len(results["radar_global"]),
        ipc_scores.notna().sum(),
        trend_scores.notna().sum(),
        results["composite_weights"]["alpha"],
        results["composite_weights"]["beta"],
        results["composite_weights"]["gamma"],
    ],
})

display(style_table(base_summary))

2026-06-17 11:27:22 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 | Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-17 11:27:22 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 | Control de antigüedad aplicado: reference_year=2026, max_age=6, celdas stale=51
2026-06-17 11:27:22 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 | Variables con más datos stale: {'public_debt_gdp': 8, 'bank_capital_rwa': 6, 'bank_concentration_5': 5, 'stock_market_cap_gdp': 5, 'financial_system_deposits_gdp': 4, 'interest_rate_spread': 4, 'atms_per_100k_adults': 3, 'current_account_gdp': 2, 'ict_goods_exports_pct_total_goods_exports': 2, 'domestic_credit_private_gdp': 2, 'digital_payment_adults_pct': 2, 'personal_remittances_gdp': 2, 'trade_openness': 2, 'commercial_bank_branches_per_100k_adults': 1, 'gdp_per_capita_ppp': 1}
2026-06-17 11:27:22 | INFO     | src.data_preparation.cleaning:apply_freshness_filt

,métrica,valor
0,Países RADAR,24.000000
1,IPC disponible,24.000000
2,Trend disponible,24.000000
3,Alpha,0.600000
4,Beta,0.300000
5,Gamma,0.100000


**Implicación.** El ranking base es el escenario central. Monte Carlo no lo reemplaza; evalúa si sus conclusiones son estables ante incertidumbre razonable en pesos.

---

## 6. Monte Carlo TOPSIS mide robustez estructural por línea

In [6]:
mc_cfg = configs["settings"].get("monte_carlo", {})

mc_topsis = run_monte_carlo_topsis_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=catalog,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(mc_cfg.get("topsis", {}).get("dimension_concentration", 150)),
    variable_concentration=float(mc_cfg.get("topsis", {}).get("variable_concentration", 100)),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

mc_topsis_summary = pd.DataFrame({
    "métrica": ["Filas simuladas", "Líneas", "Países", "Simulaciones"],
    "valor": [
        mc_topsis["simulation_long"].shape[0],
        mc_topsis["simulation_long"]["business_line"].nunique(),
        mc_topsis["simulation_long"]["country_iso3"].nunique(),
        mc_topsis["simulation_long"]["simulation_id"].nunique(),
    ],
})

display(style_table(mc_topsis_summary))

2026-06-17 11:27:24 | INFO     | src.scoring.monte_carlo:run_monte_carlo_topsis_robustness:576 | País origen excluido de Monte Carlo TOPSIS: COL
2026-06-17 11:27:24 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.729 (ESP)
2026-06-17 11:27:24 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.664 (ESP)
2026-06-17 11:27:25 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.786 (USA)
2026-06-17 11:27:25 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.801 (USA)
2026-06-17 11:27:25 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.744 (CAN)
2026-06-17 11:27:25 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.757 (ESP)
2026-06-17 11:27:25 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.676 (ESP)
2026-06-17 11:27:25 | INFO     | src

,métrica,valor
0,Filas simuladas,120000
1,Líneas,5
2,Países,24
3,Simulaciones,1000


**Interpretación.** Monte Carlo TOPSIS responde si el atractivo estructural por línea es estable cuando se perturban pesos de dimensiones y variables. No incorpora IPC ni Trend.

---

## 7. Monte Carlo RADAR mide robustez de la decisión final

In [7]:
mc_radar = run_monte_carlo_radar_robustness(
    decision_matrix=decision_matrix,
    configs=configs,
    variable_catalog=catalog,
    ipc_scores=ipc_scores,
    trend_scores=trend_scores,
    n_simulations=int(mc_cfg.get("n_simulations", 1000)),
    dimension_concentration=float(mc_cfg.get("topsis", {}).get("dimension_concentration", 150)),
    variable_concentration=float(mc_cfg.get("topsis", {}).get("variable_concentration", 100)),
    composite_concentration=float(mc_cfg.get("radar", {}).get("composite_concentration", 150)),
    perturb_composite_weights=bool(mc_cfg.get("radar", {}).get("perturb_composite_weights", True)),
    random_seed=int(mc_cfg.get("random_seed", 42)),
)

mc_radar_summary = pd.DataFrame({
    "métrica": ["Filas simuladas", "Líneas", "Países", "Simulaciones"],
    "valor": [
        mc_radar["simulation_long"].shape[0],
        mc_radar["simulation_long"]["business_line"].nunique(),
        mc_radar["simulation_long"]["country_iso3"].nunique(),
        mc_radar["simulation_long"]["simulation_id"].nunique(),
    ],
})

display(style_table(mc_radar_summary))

2026-06-17 11:30:49 | INFO     | src.scoring.monte_carlo:run_monte_carlo_radar_robustness:739 | País origen excluido de Monte Carlo RADAR: COL
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.710 (USA)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.676 (ESP)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.826 (USA)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.801 (USA)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.744 (CAN)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.769 (USA)
2026-06-17 11:30:49 | INFO     | src.scoring.ranking:rank:133 | TOPSIS completado: 24 paises | score max=0.660 (ESP)
2026-06-17 11:30:49 | INFO     | src.s

,métrica,valor
0,Filas simuladas,120000
1,Líneas,5
2,Países,24
3,Simulaciones,1000


In [8]:
expected_rows = (
    int(mc_cfg.get("n_simulations", 1000))
    * mc_radar["simulation_long"]["business_line"].nunique()
    * mc_radar["simulation_long"]["country_iso3"].nunique()
)

if mc_radar["simulation_long"].shape[0] != expected_rows:
    risk_box(
        "Revisar tamaño de simulación",
        f"Se esperaban {expected_rows:,} filas y se obtuvieron {mc_radar['simulation_long'].shape[0]:,}."
    )
else:
    success_box(
        "Monte Carlo RADAR completado",
        f"La simulación produjo {expected_rows:,} observaciones país-línea-escenario."
    )

**Lectura ejecutiva.** Monte Carlo RADAR es la prueba principal de robustez porque evalúa la métrica final usada para decisión: TOPSIS + IPC + Trend.

---

## 8. Ranking medio y volatilidad: países robustos vs sensibles

In [9]:
radar_rank_robustness = mc_radar["rank_robustness"]

def classify_rank_robustness(row: pd.Series) -> str:
    if row.get("prob_top_5", 0) >= 0.80 and row.get("Alta", 0) >= 0.70:
        return "Prioridad robusta"
    if row.get("prob_top_10", 0) >= 0.70 and (row.get("Alta", 0) + row.get("Media-alta", 0)) >= 0.70:
        return "Prioridad condicional"
    if row.get("std_rank", 999) >= 4:
        return "Alta sensibilidad"
    return "Prioridad no robusta"

radar_topn = mc_radar["topn_probabilities"]
radar_tiers = mc_radar["tier_probabilities"]

radar_robustness_exec = (
    radar_rank_robustness
    .merge(radar_topn, on=["business_line", "country_iso3"], how="left")
    .merge(radar_tiers, on=["business_line", "country_iso3"], how="left")
)

radar_robustness_exec["robustness_class"] = radar_robustness_exec.apply(classify_rank_robustness, axis=1)

display(style_table(
    radar_robustness_exec.query("business_line == 'PF'").sort_values("mean_rank").head(15),
    gradient_cols=["mean_rank", "std_rank", "prob_top_5", "Alta"],
    gradient_cmap="RdYlGn",
    format_dict={
        "mean_rank": "{:.2f}",
        "median_rank": "{:.2f}",
        "std_rank": "{:.2f}",
        "p10_rank": "{:.1f}",
        "p90_rank": "{:.1f}",
        "prob_top_3": "{:.1%}",
        "prob_top_5": "{:.1%}",
        "prob_top_10": "{:.1%}",
        "Alta": "{:.1%}",
        "Media-alta": "{:.1%}",
        "Media": "{:.1%}",
        "Baja": "{:.1%}",
    },
))

,business_line,country_iso3,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja,robustness_class
96,PF,PAN,1.29,1.00,0.51,1,4,1.0,2.0,0.663924,0.024872,0.633822,0.695042,99.5%,100.0%,100.0%,1.000000,100.0%,0.0%,0.0%,0.0%,Prioridad robusta
97,PF,ESP,2.16,2.00,0.96,1,9,1.0,4.0,0.643736,0.023922,0.613469,0.673456,89.6%,99.9%,100.0%,1.000000,99.4%,0.6%,0.0%,0.0%,Prioridad robusta
98,PF,CRI,2.78,3.00,0.50,1,4,2.0,3.0,0.631678,0.017500,0.609267,0.653800,96.1%,100.0%,100.0%,1.000000,100.0%,0.0%,0.0%,0.0%,Prioridad robusta
99,PF,DOM,3.95,4.00,0.77,1,9,3.0,5.0,0.600863,0.018229,0.578481,0.625017,14.1%,98.3%,100.0%,1.000000,86.6%,13.4%,0.0%,0.0%,Prioridad robusta
100,PF,CHL,5.36,5.00,1.31,2,12,4.0,7.0,0.561874,0.027442,0.527129,0.598211,0.6%,78.0%,99.2%,1.000000,13.1%,83.1%,3.8%,0.0%,Prioridad condicional
101,PF,URY,7.74,7.00,2.03,4,15,6.0,11.0,0.525623,0.019211,0.500894,0.549524,0.0%,1.8%,87.2%,1.000000,0.1%,82.7%,16.8%,0.4%,Prioridad condicional
102,PF,ECU,7.99,7.00,2.12,5,19,6.0,11.0,0.522853,0.024972,0.490742,0.555391,0.0%,4.5%,87.0%,0.997000,0.0%,76.1%,23.2%,0.7%,Prioridad condicional
103,PF,SLV,8.36,7.00,2.86,3,17,5.0,12.0,0.520486,0.027763,0.483899,0.557050,0.1%,15.7%,73.3%,0.994000,0.8%,67.3%,29.2%,2.7%,Prioridad no robusta
104,PF,USA,10.62,10.00,3.63,5,18,6.0,16.0,0.505916,0.027453,0.472757,0.542116,0.0%,1.1%,54.2%,0.836000,0.0%,45.9%,32.1%,22.0%,Prioridad no robusta
105,PF,PER,10.68,11.00,1.82,6,18,9.0,13.0,0.503650,0.018575,0.480751,0.527559,0.0%,0.0%,49.7%,0.981000,0.0%,25.2%,70.3%,4.5%,Prioridad no robusta


**Interpretación.** Un país robusto no es solo un país con buen rank base. Es un país con alta probabilidad de conservar posiciones superiores bajo cambios razonables de pesos.

---

## 9. Probabilidad Top-N: métrica ejecutiva superior al ranking puntual

In [10]:
for business_line in sorted(radar_topn["business_line"].unique()):
    display(Markdown(f"### {business_line} — Probabilidad Top-N RADAR"))
    display(style_table(
        radar_topn.query("business_line == @business_line").sort_values("prob_top_5", ascending=False).head(15),
        gradient_cols=["prob_top_3", "prob_top_5", "prob_top_10", "prob_top_15"],
        gradient_cmap="RdYlGn",
        format_dict={
            "prob_top_3": "{:.1%}",
            "prob_top_5": "{:.1%}",
            "prob_top_10": "{:.1%}",
            "prob_top_15": "{:.1%}",
        },
    ))

### AD — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
0,AD,CRI,96.8%,100.0%,100.0%,100.0%
1,AD,ESP,95.2%,100.0%,100.0%,100.0%
2,AD,PAN,90.8%,99.1%,100.0%,100.0%
3,AD,USA,16.5%,93.0%,100.0%,100.0%
4,AD,CHL,0.5%,83.4%,100.0%,100.0%
5,AD,DOM,0.1%,22.8%,100.0%,100.0%
6,AD,CAN,0.1%,0.9%,98.3%,100.0%
7,AD,URY,0.0%,0.8%,100.0%,100.0%
16,AD,JAM,0.0%,0.0%,0.0%,0.5%
22,AD,SUR,0.0%,0.0%,0.0%,0.0%


### BD — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
24,BD,CRI,97.6%,100.0%,100.0%,100.0%
25,BD,ESP,98.8%,100.0%,100.0%,100.0%
26,BD,PAN,92.2%,99.3%,100.0%,100.0%
27,BD,DOM,2.9%,87.7%,100.0%,100.0%
28,BD,CHL,4.1%,81.8%,100.0%,100.0%
29,BD,USA,4.4%,30.0%,98.5%,99.9%
30,BD,ECU,0.0%,1.0%,83.3%,100.0%
31,BD,MEX,0.0%,0.2%,82.8%,100.0%
40,BD,JAM,0.0%,0.0%,0.0%,0.0%
46,BD,TTO,0.0%,0.0%,0.0%,0.0%


### CIB — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
48,CIB,ESP,94.8%,100.0%,100.0%,100.0%
49,CIB,PAN,99.4%,99.9%,100.0%,100.0%
50,CIB,DOM,92.4%,99.7%,100.0%,100.0%
51,CIB,CHL,8.5%,92.1%,100.0%,100.0%
52,CIB,CRI,3.3%,51.0%,98.8%,100.0%
53,CIB,USA,0.9%,28.5%,93.5%,99.6%
54,CIB,ECU,0.7%,25.8%,99.3%,100.0%
55,CIB,CAN,0.0%,1.5%,69.5%,92.6%
56,CIB,BHS,0.0%,1.2%,37.8%,78.5%
57,CIB,MEX,0.0%,0.3%,83.2%,100.0%


### IB — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
72,IB,CRI,99.6%,100.0%,100.0%,100.0%
74,IB,PAN,99.9%,100.0%,100.0%,100.0%
73,IB,ESP,97.9%,100.0%,100.0%,100.0%
75,IB,CHL,0.0%,81.4%,100.0%,100.0%
76,IB,DOM,2.1%,70.6%,99.4%,100.0%
77,IB,USA,0.5%,22.6%,89.7%,99.7%
78,IB,ECU,0.0%,14.0%,83.5%,100.0%
79,IB,HND,0.0%,6.4%,86.8%,100.0%
80,IB,BHS,0.0%,4.7%,46.0%,94.9%
81,IB,URY,0.0%,0.3%,61.6%,100.0%


### PF — Probabilidad Top-N RADAR

,business_line,country_iso3,prob_top_3,prob_top_5,prob_top_10,prob_top_15
96,PF,CRI,96.1%,100.0%,100.0%,100.0%
97,PF,PAN,99.5%,100.0%,100.0%,100.0%
98,PF,ESP,89.6%,99.9%,100.0%,100.0%
99,PF,DOM,14.1%,98.3%,100.0%,100.0%
100,PF,CHL,0.6%,78.0%,99.2%,100.0%
101,PF,SLV,0.1%,15.7%,73.3%,99.4%
102,PF,ECU,0.0%,4.5%,87.0%,99.7%
103,PF,URY,0.0%,1.8%,87.2%,100.0%
104,PF,USA,0.0%,1.1%,54.2%,83.6%
105,PF,HND,0.0%,0.7%,32.2%,79.3%


**Implicación.** Para comité ejecutivo, `prob_top_5` y `prob_top_10` son más defendibles que una posición puntual, especialmente cuando existen empates prácticos.

---

## 10. Probabilidad por banda: estabilidad de atractividad, no solo de posición

In [11]:
for business_line in sorted(radar_tiers["business_line"].unique()):
    display(Markdown(f"### {business_line} — Probabilidad por banda RADAR"))
    display(style_table(
        radar_tiers.query("business_line == @business_line").sort_values(["Alta", "Media-alta"], ascending=False).head(15),
        gradient_cols=["Alta", "Media-alta", "Media", "Baja"],
        gradient_cmap="RdYlGn",
        format_dict={"Alta": "{:.1%}", "Media-alta": "{:.1%}", "Media": "{:.1%}", "Baja": "{:.1%}"},
    ))

### AD — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
10,AD,ESP,100.0%,0.0%,0.0%,0.0%
7,AD,CRI,99.8%,0.2%,0.0%,0.0%
16,AD,PAN,97.9%,2.1%,0.0%,0.0%
23,AD,USA,79.9%,20.1%,0.0%,0.0%
8,AD,DOM,11.7%,88.2%,0.1%,0.0%
6,AD,CHL,10.5%,89.5%,0.0%,0.0%
5,AD,CAN,0.2%,97.3%,2.5%,0.0%
22,AD,URY,0.0%,100.0%,0.0%,0.0%
1,AD,BHS,0.0%,66.1%,31.6%,2.3%
17,AD,PER,0.0%,25.0%,75.0%,0.0%


### BD — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
31,BD,CRI,100.0%,0.0%,0.0%,0.0%
34,BD,ESP,100.0%,0.0%,0.0%,0.0%
40,BD,PAN,96.8%,3.2%,0.0%,0.0%
32,BD,DOM,66.9%,33.1%,0.0%,0.0%
30,BD,CHL,23.2%,76.8%,0.0%,0.0%
47,BD,USA,13.1%,84.2%,2.6%,0.1%
33,BD,ECU,0.0%,72.5%,27.3%,0.2%
38,BD,MEX,0.0%,66.6%,33.4%,0.0%
46,BD,URY,0.0%,63.4%,36.5%,0.1%
24,BD,ARG,0.0%,51.4%,47.9%,0.7%


### CIB — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
58,CIB,ESP,100.0%,0.0%,0.0%,0.0%
64,CIB,PAN,99.7%,0.3%,0.0%,0.0%
56,CIB,DOM,98.6%,1.4%,0.0%,0.0%
54,CIB,CHL,61.7%,38.3%,0.0%,0.0%
55,CIB,CRI,29.9%,66.7%,3.4%,0.0%
71,CIB,USA,5.6%,85.7%,7.9%,0.8%
57,CIB,ECU,3.8%,92.6%,3.6%,0.0%
49,CIB,BHS,0.5%,29.0%,41.4%,29.1%
53,CIB,CAN,0.2%,56.7%,31.6%,11.5%
62,CIB,MEX,0.0%,66.4%,33.1%,0.5%


### IB — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
88,IB,PAN,100.0%,0.0%,0.0%,0.0%
79,IB,CRI,99.9%,0.1%,0.0%,0.0%
82,IB,ESP,99.9%,0.1%,0.0%,0.0%
80,IB,DOM,45.3%,53.7%,1.0%,0.0%
78,IB,CHL,43.9%,56.0%,0.1%,0.0%
95,IB,USA,5.8%,75.9%,17.6%,0.7%
81,IB,ECU,2.7%,72.6%,24.7%,0.0%
73,IB,BHS,1.4%,35.0%,54.7%,8.9%
84,IB,HND,1.1%,72.3%,26.6%,0.0%
91,IB,SLV,0.0%,66.1%,33.9%,0.0%


### PF — Probabilidad por banda RADAR

tier,business_line,country_iso3,Alta,Media-alta,Media,Baja
103,PF,CRI,100.0%,0.0%,0.0%,0.0%
112,PF,PAN,100.0%,0.0%,0.0%,0.0%
106,PF,ESP,99.4%,0.6%,0.0%,0.0%
104,PF,DOM,86.6%,13.4%,0.0%,0.0%
102,PF,CHL,13.1%,83.1%,3.8%,0.0%
115,PF,SLV,0.8%,67.3%,29.2%,2.7%
118,PF,URY,0.1%,82.7%,16.8%,0.4%
105,PF,ECU,0.0%,76.1%,23.2%,0.7%
119,PF,USA,0.0%,45.9%,32.1%,22.0%
96,PF,ARG,0.0%,43.8%,41.5%,14.7%


**Lectura ejecutiva.** Un país puede no tener alta probabilidad de Top 5, pero sí alta probabilidad de permanecer en banda alta o media-alta. Esa distinción es clave para priorización exploratoria.

---

## 11. Estabilidad de correlaciones entre líneas: la diferenciación debe sobrevivir al Monte Carlo

In [12]:
line_corr_radar = mc_radar["line_correlation_robustness"]

display(style_table(
    line_corr_radar,
    gradient_cols=["mean_spearman", "p10_spearman", "p90_spearman"],
    gradient_cmap="YlOrRd",
    format_dict={
        "mean_spearman": "{:.3f}",
        "median_spearman": "{:.3f}",
        "std_spearman": "{:.3f}",
        "p10_spearman": "{:.3f}",
        "p90_spearman": "{:.3f}",
        "min_spearman": "{:.3f}",
        "max_spearman": "{:.3f}",
    },
))

,line_a,line_b,mean_spearman,median_spearman,std_spearman,p10_spearman,p90_spearman,min_spearman,max_spearman
0,IB,CIB,0.879,0.883,0.041,0.823,0.925,0.718,0.977
1,AD,BD,0.868,0.870,0.031,0.827,0.906,0.761,0.957
2,PF,BD,0.859,0.889,0.091,0.727,0.949,0.457,0.981
3,BD,CIB,0.824,0.824,0.045,0.763,0.880,0.660,0.957
4,IB,PF,0.823,0.830,0.056,0.749,0.887,0.544,0.963
5,PF,CIB,0.803,0.804,0.058,0.729,0.873,0.560,0.983
6,AD,CIB,0.802,0.804,0.055,0.730,0.873,0.623,0.948
7,PF,AD,0.784,0.812,0.090,0.657,0.874,0.439,0.924
8,IB,BD,0.775,0.780,0.060,0.692,0.846,0.573,0.951
9,IB,AD,0.773,0.775,0.068,0.685,0.858,0.555,0.939


In [13]:
pf_bd_corr = line_corr_radar[
    ((line_corr_radar["line_a"] == "PF") & (line_corr_radar["line_b"] == "BD"))
    | ((line_corr_radar["line_a"] == "BD") & (line_corr_radar["line_b"] == "PF"))
]

display(style_table(pf_bd_corr, format_dict={col: "{:.3f}" for col in pf_bd_corr.select_dtypes(include="number").columns}))

,line_a,line_b,mean_spearman,median_spearman,std_spearman,p10_spearman,p90_spearman,min_spearman,max_spearman
2,PF,BD,0.859,0.889,0.091,0.727,0.949,0.457,0.981


**Interpretación.** Si PF–BD mantiene correlación alta en Monte Carlo, la cercanía entre pagos y banca digital es estructural, no un artefacto de pesos. Si la correlación se vuelve inestable, la diferenciación depende demasiado de la parametrización.

---

## 12. Distribución de alpha, beta y gamma: la mezcla RADAR también es un supuesto

In [14]:
composite_distribution = mc_radar["composite_weight_distribution"]
display(style_table(composite_distribution))

,alpha,beta,gamma
mean,0.599872,0.301225,0.098902
std,0.040086,0.035993,0.025305
min,0.467503,0.191124,0.029930
max,0.737372,0.423866,0.195620


**Lectura ejecutiva.** La simulación perturba `alpha`, `beta` y `gamma` alrededor de la mezcla base. Esto permite evaluar si la decisión final depende excesivamente de la ponderación entre estructura, proximidad y tendencia.

---

## 13. Tabla ejecutiva final por línea: ranking base + robustez Monte Carlo

In [15]:
def build_base_radar_by_line(
    radar_by_line,
    countries_eval: Optional[List[str]] = None,
    business_lines: Optional[List[str]] = None,
) -> Dict[str, pd.DataFrame]:
    """Convierte results['radar_by_line'] en DataFrames por línea."""

    out: Dict[str, pd.DataFrame] = {}

    if isinstance(radar_by_line, pd.DataFrame):
        df = radar_by_line.copy()
        if "country_iso3" not in df.columns:
            df = df.reset_index()
            if "index" in df.columns:
                df = df.rename(columns={"index": "country_iso3"})
        if "country_iso3" not in df.columns:
            raise ValueError("radar_by_line no contiene country_iso3.")

        df["country_iso3"] = df["country_iso3"].astype(str).str.strip()

        if business_lines is None:
            business_lines = [col for col in df.columns if col != "country_iso3"]

        for business_line in business_lines:
            if business_line not in df.columns:
                continue
            tmp = df[["country_iso3", business_line]].copy()
            tmp = tmp.rename(columns={business_line: "radar_score"})
            tmp["radar_score"] = pd.to_numeric(tmp["radar_score"], errors="coerce")
            tmp = tmp.dropna(subset=["radar_score"])
            tmp["rank"] = tmp["radar_score"].rank(ascending=False, method="min").astype(int)
            out[business_line] = tmp.sort_values("rank").reset_index(drop=True)

        return out

    if isinstance(radar_by_line, dict):
        for business_line, obj in radar_by_line.items():
            if business_line == "country_iso3":
                continue

            if isinstance(obj, pd.DataFrame):
                tmp = obj.copy()
                if "country_iso3" not in tmp.columns:
                    tmp = tmp.reset_index()
                    if "index" in tmp.columns:
                        tmp = tmp.rename(columns={"index": "country_iso3"})
                score_col = next((col for col in ["radar_score", "score", "final_score", business_line, "value"] if col in tmp.columns), None)
                if score_col is None:
                    raise ValueError(f"No se pudo inferir score para {business_line}.")
                tmp = tmp[["country_iso3", score_col]].rename(columns={score_col: "radar_score"})

            elif isinstance(obj, pd.Series):
                if not pd.api.types.is_integer_dtype(obj.index):
                    tmp = obj.rename("radar_score").reset_index()
                    if tmp.columns[0] != "country_iso3":
                        tmp = tmp.rename(columns={tmp.columns[0]: "country_iso3"})
                else:
                    if countries_eval is None:
                        raise ValueError(f"{business_line} es Series con índice numérico; pase countries_eval o corrija results['radar_by_line'].")
                    if len(obj) != len(countries_eval):
                        raise ValueError(f"Longitud incompatible para {business_line}.")
                    tmp = pd.DataFrame({"country_iso3": countries_eval, "radar_score": obj.values})
            else:
                raise TypeError(f"Tipo no soportado para {business_line}: {type(obj)}")

            tmp["country_iso3"] = tmp["country_iso3"].astype(str).str.strip()
            tmp["radar_score"] = pd.to_numeric(tmp["radar_score"], errors="coerce")
            tmp = tmp.dropna(subset=["radar_score"])
            tmp["rank"] = tmp["radar_score"].rank(ascending=False, method="min").astype(int)
            out[business_line] = tmp.sort_values("rank").reset_index(drop=True)

        return out

    raise TypeError(f"Tipo no soportado para radar_by_line: {type(radar_by_line)}")


def build_mc_executive_table(
    base_ranking: Union[pd.DataFrame, pd.Series],
    mc_rank_robustness: pd.DataFrame,
    mc_topn: pd.DataFrame,
    mc_tiers: pd.DataFrame,
    business_line: str,
    score_col: Optional[str] = "radar_score",
) -> pd.DataFrame:
    """Construye tabla ejecutiva de robustez Monte Carlo para una línea."""

    base = base_ranking.copy()

    if isinstance(base, pd.Series):
        if pd.api.types.is_integer_dtype(base.index):
            raise ValueError("base_ranking es Series con índice numérico. Use DataFrame con country_iso3.")
        base = base.rename("radar_score").reset_index()
        if base.columns[0] != "country_iso3":
            base = base.rename(columns={base.columns[0]: "country_iso3"})

    if "country_iso3" not in base.columns:
        base = base.reset_index()
        if "index" in base.columns:
            base = base.rename(columns={"index": "country_iso3"})

    if score_col not in base.columns:
        candidate = next((col for col in ["radar_score", "score", "final_score", "value"] if col in base.columns), None)
        if candidate is None:
            raise ValueError(f"No se pudo inferir score. Columnas: {list(base.columns)}")
        score_col = candidate

    base = base[["country_iso3", score_col]].rename(columns={score_col: "base_radar_score"})
    base["country_iso3"] = base["country_iso3"].astype(str).str.strip()
    base["base_rank"] = base["base_radar_score"].rank(ascending=False, method="min").astype(int)

    robustness = mc_rank_robustness.query("business_line == @business_line").copy()
    topn = mc_topn.query("business_line == @business_line").copy()
    tiers = mc_tiers.query("business_line == @business_line").copy()

    for df in [robustness, topn, tiers]:
        df["country_iso3"] = df["country_iso3"].astype(str).str.strip()
        df["business_line"] = df["business_line"].astype(str).str.strip()

    out = (
        base
        .merge(robustness, on="country_iso3", how="left")
        .merge(topn, on=["business_line", "country_iso3"], how="left")
        .merge(tiers, on=["business_line", "country_iso3"], how="left")
    )

    return out.sort_values("base_rank").reset_index(drop=True)

base_radar_by_line = build_base_radar_by_line(results["radar_by_line"])

mc_exec_by_line = {}
for business_line, base_df in base_radar_by_line.items():
    mc_exec_by_line[business_line] = build_mc_executive_table(
        base_ranking=base_df,
        mc_rank_robustness=mc_radar["rank_robustness"],
        mc_topn=mc_radar["topn_probabilities"],
        mc_tiers=mc_radar["tier_probabilities"],
        business_line=business_line,
        score_col="radar_score",
    )

for business_line, exec_table in mc_exec_by_line.items():
    display(Markdown(f"### {business_line} — Tabla ejecutiva Monte Carlo RADAR"))
    display(style_table(
        exec_table.head(15),
        gradient_cols=["base_radar_score", "mean_rank", "std_rank", "prob_top_5", "Alta"],
        gradient_cmap="RdYlGn",
        format_dict={
            "base_radar_score": "{:.3f}",
            "base_rank": "{:.0f}",
            "mean_rank": "{:.2f}",
            "median_rank": "{:.2f}",
            "std_rank": "{:.2f}",
            "p10_rank": "{:.1f}",
            "p90_rank": "{:.1f}",
            "prob_top_3": "{:.1%}",
            "prob_top_5": "{:.1%}",
            "prob_top_10": "{:.1%}",
            "prob_top_15": "{:.1%}",
            "Alta": "{:.1%}",
            "Media-alta": "{:.1%}",
            "Media": "{:.1%}",
            "Baja": "{:.1%}",
        },
    ))

### IB — Tabla ejecutiva Monte Carlo RADAR

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,PAN,0.713,1,IB,1.03,1.00,0.21,1,4,1.0,1.0,0.712401,0.023545,0.682371,0.743848,99.9%,100.0%,100.0%,100.0%,100.0%,0.0%,0.0%,0.0%
1,CRI,0.666,2,IB,2.36,2.00,0.51,1,5,2.0,3.0,0.666846,0.019729,0.641167,0.692232,99.6%,100.0%,100.0%,100.0%,99.9%,0.1%,0.0%,0.0%
2,ESP,0.662,3,IB,2.63,3.00,0.56,1,5,2.0,3.0,0.659904,0.017806,0.637650,0.682669,97.9%,100.0%,100.0%,100.0%,99.9%,0.1%,0.0%,0.0%
3,CHL,0.595,4,IB,4.84,5.00,0.98,4,10,4.0,6.0,0.595264,0.024763,0.563042,0.627403,0.0%,81.4%,100.0%,100.0%,43.9%,56.0%,0.1%,0.0%
4,DOM,0.593,5,IB,5.05,5.00,1.37,3,13,4.0,7.0,0.595259,0.022340,0.566941,0.624887,2.1%,70.6%,99.4%,100.0%,45.3%,53.7%,1.0%,0.0%
5,USA,0.567,6,IB,7.30,7.00,2.27,3,16,5.0,11.0,0.566637,0.023174,0.538396,0.595619,0.5%,22.6%,89.7%,99.7%,5.8%,75.9%,17.6%,0.7%
6,ECU,0.560,7,IB,7.80,7.00,2.29,4,14,5.0,11.0,0.560368,0.028971,0.522497,0.596892,0.0%,14.0%,83.5%,100.0%,2.7%,72.6%,24.7%,0.0%
7,HND,0.558,8,IB,8.10,8.00,1.91,4,13,6.0,11.0,0.555673,0.024584,0.522708,0.587283,0.0%,6.4%,86.8%,100.0%,1.1%,72.3%,26.6%,0.0%
8,SLV,0.550,9,IB,9.05,9.00,1.19,6,13,8.0,11.0,0.549371,0.020149,0.523130,0.575167,0.0%,0.0%,87.3%,100.0%,0.0%,66.1%,33.9%,0.0%
9,URY,0.544,10,IB,9.64,10.00,2.20,5,15,7.0,13.0,0.544570,0.016846,0.523544,0.567187,0.0%,0.3%,61.6%,100.0%,0.0%,45.8%,53.8%,0.4%


### PF — Tabla ejecutiva Monte Carlo RADAR

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,PAN,0.663,1,PF,1.29,1.00,0.51,1,4,1.0,2.0,0.663924,0.024872,0.633822,0.695042,99.5%,100.0%,100.0%,100.0%,100.0%,0.0%,0.0%,0.0%
1,ESP,0.645,2,PF,2.16,2.00,0.96,1,9,1.0,4.0,0.643736,0.023922,0.613469,0.673456,89.6%,99.9%,100.0%,100.0%,99.4%,0.6%,0.0%,0.0%
2,CRI,0.632,3,PF,2.78,3.00,0.50,1,4,2.0,3.0,0.631678,0.017500,0.609267,0.653800,96.1%,100.0%,100.0%,100.0%,100.0%,0.0%,0.0%,0.0%
3,DOM,0.600,4,PF,3.95,4.00,0.77,1,9,3.0,5.0,0.600863,0.018229,0.578481,0.625017,14.1%,98.3%,100.0%,100.0%,86.6%,13.4%,0.0%,0.0%
4,CHL,0.563,5,PF,5.36,5.00,1.31,2,12,4.0,7.0,0.561874,0.027442,0.527129,0.598211,0.6%,78.0%,99.2%,100.0%,13.1%,83.1%,3.8%,0.0%
5,URY,0.525,6,PF,7.74,7.00,2.03,4,15,6.0,11.0,0.525623,0.019211,0.500894,0.549524,0.0%,1.8%,87.2%,100.0%,0.1%,82.7%,16.8%,0.4%
6,ECU,0.521,7,PF,7.99,7.00,2.12,5,19,6.0,11.0,0.522853,0.024972,0.490742,0.555391,0.0%,4.5%,87.0%,99.7%,0.0%,76.1%,23.2%,0.7%
7,SLV,0.519,8,PF,8.36,7.00,2.86,3,17,5.0,12.0,0.520486,0.027763,0.483899,0.557050,0.1%,15.7%,73.3%,99.4%,0.8%,67.3%,29.2%,2.7%
8,USA,0.507,9,PF,10.62,10.00,3.63,5,18,6.0,16.0,0.505916,0.027453,0.472757,0.542116,0.0%,1.1%,54.2%,83.6%,0.0%,45.9%,32.1%,22.0%
9,ARG,0.507,10,PF,10.75,10.00,2.89,6,19,8.0,15.0,0.505128,0.024219,0.475177,0.536549,0.0%,0.0%,60.8%,92.4%,0.0%,43.8%,41.5%,14.7%


### AD — Tabla ejecutiva Monte Carlo RADAR

...

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,PAN,0.668,1,CIB,1.08,1.00,0.38,1,6,1.0,1.0,0.669294,0.023317,0.640539,0.700053,99.4%,99.9%,100.0%,100.0%,99.7%,0.3%,0.0%,0.0%
1,DOM,0.627,2,CIB,2.53,2.00,0.73,1,6,2.0,3.0,0.626403,0.019489,0.601042,0.651418,92.4%,99.7%,100.0%,100.0%,98.6%,1.4%,0.0%,0.0%
2,ESP,0.626,3,CIB,2.56,3.00,0.64,1,4,2.0,3.0,0.625343,0.011799,0.609819,0.640207,94.8%,100.0%,100.0%,100.0%,100.0%,0.0%,0.0%,0.0%
3,CHL,0.588,4,CIB,4.38,4.00,0.80,1,8,4.0,5.0,0.587611,0.021037,0.560522,0.614722,8.5%,92.1%,100.0%,100.0%,61.7%,38.3%,0.0%,0.0%
4,CRI,0.564,5,CIB,5.72,5.00,1.74,2,13,4.0,8.0,0.568160,0.027582,0.533123,0.603982,3.3%,51.0%,98.8%,100.0%,29.9%,66.7%,3.4%,0.0%
5,ECU,0.555,6,CIB,6.46,6.00,1.46,3,12,5.0,9.0,0.555839,0.025575,0.523180,0.589921,0.7%,25.8%,99.3%,100.0%,3.8%,92.6%,3.6%,0.0%
6,USA,0.553,7,CIB,6.81,7.00,2.13,1,18,5.0,9.0,0.552527,0.024541,0.522228,0.584657,0.9%,28.5%,93.5%,99.6%,5.6%,85.7%,7.9%,0.8%
7,CAN,0.528,8,CIB,9.72,9.00,3.18,4,19,6.0,15.0,0.526698,0.025403,0.494822,0.559955,0.0%,1.5%,69.5%,92.6%,0.2%,56.7%,31.6%,11.5%
8,MEX,0.525,9,CIB,9.04,9.00,1.68,5,15,7.0,11.0,0.525224,0.016246,0.503570,0.546685,0.0%,0.3%,83.2%,100.0%,0.0%,66.4%,33.1%,0.5%
9,PER,0.513,10,CIB,10.56,10.00,1.80,6,17,8.0,13.0,0.514096,0.019123,0.488170,0.537753,0.0%,0.0%,55.1%,99.2%,0.0%,28.4%,69.2%,2.4%


### GLOBAL — Tabla ejecutiva Monte Carlo RADAR

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,PAN,0.691,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
1,CRI,0.653,2,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
2,ESP,0.627,3,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
3,DOM,0.620,4,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
4,CHL,0.569,5,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
5,USA,0.552,6,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
6,URY,0.539,7,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
7,ECU,0.538,8,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
8,MEX,0.533,9,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
9,GTM,0.530,10,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%


### rank_global — Tabla ejecutiva Monte Carlo RADAR

,country_iso3,base_radar_score,base_rank,business_line,mean_rank,median_rank,std_rank,min_rank,max_rank,p10_rank,p90_rank,mean_score,std_score,p10_score,p90_score,prob_top_3,prob_top_5,prob_top_10,prob_top_15,Alta,Media-alta,Media,Baja
0,SUR,24.000,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
1,BLZ,23.000,2,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
2,TTO,22.000,3,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
3,BRA,21.000,4,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
4,JAM,20.000,5,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
5,BOL,19.000,6,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
6,ARG,18.000,7,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
7,NIC,17.000,8,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
8,PRY,16.000,9,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%
9,BHS,15.000,10,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan%,nan%,nan%,nan%,nan%,nan%,nan%,nan%


**Interpretación.** Esta tabla conecta el ranking base con su robustez probabilística. Es el output más útil para comité porque traduce posiciones puntuales en probabilidades de mantenerse en bandas o Top-N.

---

## 14. Países robustos, condicionales y sensibles

In [16]:
robustness_counts = (
    radar_robustness_exec
    .groupby(["business_line", "robustness_class"])
    .size()
    .rename("n_countries")
    .reset_index()
)

display(style_table(robustness_counts, gradient_cols=["n_countries"], format_dict={"n_countries": "{:,.0f}"}))

fig = px.bar(
    robustness_counts,
    x="business_line",
    y="n_countries",
    color="robustness_class",
    title="Clasificación de países según robustez Monte Carlo RADAR",
    color_discrete_map={
        "Prioridad robusta": CIBEST["green"],
        "Prioridad condicional": CIBEST["gold"],
        "Alta sensibilidad": CIBEST["amber"],
        "Prioridad no robusta": CIBEST["red"],
    },
)
fig.update_layout(xaxis_title="Línea de negocio", yaxis_title="Número de países")
fig.show()

,business_line,robustness_class,n_countries
0,AD,Prioridad condicional,4
1,AD,Prioridad no robusta,16
2,AD,Prioridad robusta,4
3,BD,Prioridad condicional,4
4,BD,Prioridad no robusta,17
5,BD,Prioridad robusta,3
6,CIB,Prioridad condicional,4
7,CIB,Prioridad no robusta,17
8,CIB,Prioridad robusta,3
9,IB,Prioridad condicional,5


**Lectura ejecutiva.** Esta clasificación evita tratar todos los países top como equivalentes. Una prioridad robusta puede avanzar a análisis comercial; una prioridad condicional requiere validación adicional de drivers, riesgos y supuestos.

---

## 15. Persistencia de resultados para trazabilidad

In [ ]:
scores_dir = resolve_data_path(configs["settings"]["data"]["scores_path"])
scores_dir.mkdir(parents=True, exist_ok=True)

mc_radar["rank_robustness"].to_parquet(scores_dir / "mc_radar_rank_robustness_latest.parquet", index=False)
mc_radar["topn_probabilities"].to_parquet(scores_dir / "mc_radar_topn_probabilities_latest.parquet", index=False)
mc_radar["tier_probabilities"].to_parquet(scores_dir / "mc_radar_tier_probabilities_latest.parquet", index=False)
mc_radar["line_correlation_robustness"].to_parquet(scores_dir / "mc_radar_line_correlation_robustness_latest.parquet", index=False)
mc_topsis["rank_robustness"].to_parquet(scores_dir / "mc_topsis_rank_robustness_latest.parquet", index=False)

for business_line, exec_table in mc_exec_by_line.items():
    exec_table.to_parquet(scores_dir / f"mc_exec_{business_line}_latest.parquet", index=False)

print(f"Resultados Monte Carlo persistidos en: {scores_dir}")

Resultados Monte Carlo persistidos en: C:\Users\jadarve\OneDrive - Grupo Bancolombia\Bancolombia\GFEC-VEF\2026\Internacionalización\radar_cibest_v2\data\scores


: 

El “error” de RADAR no debe medirse como si fuera un modelo supervisado tradicional; debe medirse como incertidumbre, estabilidad, validez y riesgo de decisión. Solo podrás calcular un error tipo MAE/RMSE si defines una variable objetivo observada —por ejemplo, rentabilidad real, éxito de entrada, volumen captado o desempeño comercial posterior—. Mientras RADAR sea un modelo multicriterio de priorización, la pregunta correcta es:

¿Qué tan confiable es el ranking para soportar decisiones bajo incertidumbre de datos, pesos, imputación, método y validación externa?

1. Primero: distinguir “error” de “incertidumbre del modelo”
En RADAR hay cuatro tipos de error/riesgo distintos:

| Tipo                         | Pregunta                                             | Cómo medirlo                                                   |
| ---------------------------- | ---------------------------------------------------- | -------------------------------------------------------------- |
| **Error de datos**           | ¿Los insumos son completos, vigentes y confiables?   | cobertura efectiva, stale data, imputación, outliers           |
| **Error de parametrización** | ¿El ranking cambia si cambian los pesos?             | Monte Carlo, sensibilidad, elasticidad de pesos                |
| **Error metodológico**       | ¿El resultado depende demasiado de TOPSIS?           | comparación TOPSIS vs VIKOR/PROMETHEE, correlación de rankings |
| **Error de decisión**        | ¿El modelo recomienda países que luego no funcionan? | backtesting, validación con expertos, desempeño ex post        |


Actualmente RADAR prioriza países con base en datos estructurales, IPC y Trend. Si todavía no tienes una variable real tipo “éxito comercial observado”, entonces no se puede decir: error = predicción - realidad.

Lo que sí se puede medir es:
riesgo de que la recomendación cambie bajo supuestos razonables


- Error por sensibilidad de pesos: Mide qué tanto cambia el ranking cuando perturbas pesos de dimensiones, variables y mezcla RADAR.

Métricas:

| Métrica                | Interpretación                           |
| ---------------------- | ---------------------------------------- |
| `mean_rank`            | posición promedio bajo simulaciones      |
| `std_rank`             | volatilidad del ranking                  |
| `p10_rank`, `p90_rank` | rango probable de posiciones             |
| `prob_top_5`           | probabilidad de estar en Top 5           |
| `prob_top_10`          | probabilidad de estar en Top 10          |
| `prob_band_alta`       | probabilidad de permanecer en banda alta |


Ejemplo de lectura:
País A:
rank base = 3
mean_rank = 3.4
std_rank = 0.8
prob_top_5 = 92%

→ Recomendación robusta.

País B:
rank base = 4
mean_rank = 8.7
std_rank = 5.2
prob_top_5 = 41%

→ Recomendación frágil.

- Error por dependencia de método multicriterio: 
Spearman(TOPSIS, VIKOR)
KendallTau(TOPSIS, VIKOR)
Top-5 overlap
Top-10 overlap

| Resultado       | Interpretación                          |
| --------------- | --------------------------------------- |
| Spearman ≥ 0.85 | alta robustez metodológica              |
| 0.70–0.85       | diferencias relevantes, pero manejables |
| < 0.70          | ranking sensible al método              |

Si un país es Top 5 en TOPSIS pero cae mucho en VIKOR, no necesariamente está mal; significa que su prioridad depende de la lógica multicriterio elegida.

- Error por imputación
* imputation_share_country = celdas imputadas país / variables evaluadas
* imputation_share_variable = países imputados variable / países evaluados

| Nivel   | Criterio sugerido | Uso                      |
| ------- | ----------------: | ------------------------ |
| Bajo    |     <10% imputado | confiable                |
| Medio   |            10–20% | usar con nota            |
| Alto    |            20–30% | país condicionado        |
| Crítico |              >30% | excluir antes de scoring |




**Implicación.** Los resultados de sensibilidad deben persistirse para reproducibilidad, comparación entre corridas y construcción posterior de visualizaciones ejecutivas.

---

## 16. Hallazgos principales

1. La robustez ejecutiva debe evaluarse sobre RADAR completo, no solo sobre TOPSIS.
2. Monte Carlo TOPSIS mide estabilidad estructural; Monte Carlo RADAR mide estabilidad de la decisión final.
3. La probabilidad Top-N y la probabilidad por banda son más útiles que el ranking puntual para decisiones de comité.
4. Países con alta volatilidad de ranking deben tratarse como prioridades condicionales, aunque aparezcan altos en el escenario base.
5. La estabilidad de correlaciones entre líneas permite saber si la diferenciación de tesis de negocio sobrevive a cambios de pesos.
6. `alpha`, `beta` y `gamma` son supuestos del modelo; simularlos evita sobreconfiar en una mezcla fija.
7. Las tablas ejecutivas por línea son el insumo principal para priorización robusta de mercados.

---

## 17. Limitaciones

- Monte Carlo perturba pesos, no datos. No captura error de medición en variables, IPC o Trend.
- Las distribuciones Dirichlet dependen de parámetros de concentración; estos deben documentarse como supuestos.
- Una probabilidad alta de Top-N no implica factibilidad comercial o regulatoria inmediata.
- La simulación evalúa robustez interna del modelo, no escenarios macroeconómicos futuros.
- TOPSIS y RADAR siguen siendo modelos relativos al conjunto de países evaluados.

---

## 18. Recomendaciones y próximos pasos

1. Presentar al comité el ranking base acompañado de `prob_top_5`, `prob_top_10` y probabilidad de banda alta.
2. Clasificar países como prioridad robusta, prioridad condicional, alta sensibilidad o no robusta.
3. Para países condicionales, revisar drivers, restricciones y riesgos cualitativos antes de recomendación final.
4. Usar estabilidad de correlaciones para justificar similitudes entre líneas como PF–BD o IB–CIB.
5. Mantener Monte Carlo RADAR como prueba estándar antes de publicar rankings ejecutivos.
6. En iteraciones futuras, evaluar perturbación de IPC y Trend solo si se cuenta con supuestos defendibles de error de medición.

---

## 19. Síntesis Ejecutiva

- El análisis de sensibilidad convierte el ranking RADAR en una lectura probabilística de robustez.
- La simulación principal debe ser Monte Carlo RADAR, porque el RADAR compuesto es la métrica de decisión.
- TOPSIS Monte Carlo es diagnóstico estructural; RADAR Monte Carlo es evidencia ejecutiva.
- Países prioritarios deben clasificarse por probabilidad de mantenerse en Top-N y banda alta, no solo por rank base.
- Las decisiones finales deben distinguir prioridades robustas de oportunidades condicionales sensibles a pesos.